# 동형암호(FHE) 원리 실습 — 순수 파이썬(설치 필요 없음)

**서울여대 GSPP Privacy Scholar Camp · 이승환 (waLLLnut / LatticA)**

`numpy` 없이 표준 라이브러리(`random`, `cmath`)만으로 돌립니다.
**토이 크기($n=2$, $N=4$)라 for-루프도 쓰지 않고 회로를 성분별로 통째로 펼쳐서** 직접 적었습니다
— 어떤 곱과 합이 일어나는지가 눈에 다 보이게. (그래서 깁니다.)

> `★ 실습` 셀의 값·슬롯을 바꿔 다시 실행해 보세요.
> **주의:** 장난감 파라미터 — 실제 보안 파라미터가 아닙니다.


## 0. 파라미터
$n=2$, $q=10^6$, $\Delta=10^2$. 비밀키는 두 성분 $s_1,s_2$.


In [ ]:
import random, cmath   # 파이썬 기본 내장

q = 10**6
Delta = 100
random.seed(42)

def center(x):
    "값을 (-q/2, q/2] 범위로 (부호 있는 대표원)"
    return ((int(x) + q // 2) % q) - q // 2

s1 = random.randint(-10, 10)   # 비밀키 성분 1
s2 = random.randint(-10, 10)   # 비밀키 성분 2
print("비밀키 s1 =", s1, ", s2 =", s2)

## 1. LWE 암호화 / phase / 복호화
암호문 $=(c_0,c_1,c_2)=(b,\,-a_1,\,-a_2)$.
**phase** $=c_0\cdot1+c_1 s_1+c_2 s_2 = b - a_1 s_1 - a_2 s_2 = \Delta m + e$.


In [ ]:
def encrypt(m):
    a1 = random.randint(0, q - 1)
    a2 = random.randint(0, q - 1)
    e  = random.randint(-10, 10)
    b  = (a1 * s1 + a2 * s2 + e + Delta * m) % q     # 회로: a1*s1 + a2*s2 + e + Δm
    return [b, (-a1) % q, (-a2) % q]                 # (c0, c1, c2)

def phase(c):
    # 회로: c0*1 + c1*s1 + c2*s2  (내적을 그대로 펼침)
    return center(c[0] + c[1] * s1 + c[2] * s2)

def decrypt(c):
    return round(phase(c) / Delta)

c1 = encrypt(2)     # 메시지 2
c2 = encrypt(3)     # 메시지 3
print("암호문 c1 =", c1)
print("phase(c1) =", phase(c1), "-> 복호:", decrypt(c1))
print("phase(c2) =", phase(c2), "-> 복호:", decrypt(c2))
assert decrypt(c1) == 2 and decrypt(c2) == 3
print("OK: 암호화/복호화")

## 2. 덧셈 — 세 성분을 그대로 더한다

In [ ]:
# 회로: 성분별 덧셈 (세 개를 직접)
c_add = [(c1[0] + c2[0]) % q,
         (c1[1] + c2[1]) % q,
         (c1[2] + c2[2]) % q]

print("phase(c1+c2) =", phase(c_add), "-> 복호:", decrypt(c_add), "(기대 5)")
assert decrypt(c_add) == 2 + 3
print("OK: 동형 덧셈")

### ★ 실습 1. 메시지와 오류를 바꾸면?
세 값만 바꿔 실행. 오류 한계가 $\Delta/2=50$ 에 가까워지면 언제 복호가 흔들릴까요?


In [ ]:
TRY_M1, TRY_M2 = 4, -1
TRY_ERROR_BOUND = 10

trng = random.Random(2026)

def encrypt_trial(m, bound):
    a1 = trng.randint(0, q - 1)
    a2 = trng.randint(0, q - 1)
    e  = trng.randint(-bound, bound)
    b  = (a1 * s1 + a2 * s2 + e + Delta * m) % q
    return [b, (-a1) % q, (-a2) % q], e

t1, e1 = encrypt_trial(TRY_M1, TRY_ERROR_BOUND)
t2, e2 = encrypt_trial(TRY_M2, TRY_ERROR_BOUND)
tadd = [(t1[0] + t2[0]) % q, (t1[1] + t2[1]) % q, (t1[2] + t2[2]) % q]

g1, g2, gadd = decrypt(t1), decrypt(t2), decrypt(tadd)
print("오류 (e1, e2):", (e1, e2), " / 합:", e1 + e2, " (기준 |합| <", Delta // 2, ")")
print("개별 복호:", (g1, g2), " / 예상:", (TRY_M1, TRY_M2))
print("덧셈 복호:", gadd, " / 예상:", TRY_M1 + TRY_M2)
if (g1, g2, gadd) == (TRY_M1, TRY_M2, TRY_M1 + TRY_M2):
    print("결과: 성공")
else:
    print("결과: 복호 실패 — 오류와 Delta의 상대적 크기를 확인하세요")

## 3. 곱셈 — 텐서곱 (9성분 모두 직접)
확장키 $\bar{\mathbf s}=(1,s_1,s_2)$. 두 phase 의 곱이
$\langle\bar{\mathbf c}_1\otimes\bar{\mathbf c}_2,\ \bar{\mathbf s}\otimes\bar{\mathbf s}\rangle$.
키·암호문 텐서곱(각 $3\times3=9$ 성분)과 그 내적(9항)을 전부 펼칩니다.


In [ ]:
# 키 텐서곱  sbar ⊗ sbar   (sbar = [1, s1, s2])
t = [1 * 1,   1 * s1,   1 * s2,
     s1 * 1,  s1 * s1,  s1 * s2,
     s2 * 1,  s2 * s1,  s2 * s2]

# 암호문 텐서곱  c1 ⊗ c2   (mod q)
cm = [(c1[0] * c2[0]) % q, (c1[0] * c2[1]) % q, (c1[0] * c2[2]) % q,
      (c1[1] * c2[0]) % q, (c1[1] * c2[1]) % q, (c1[1] * c2[2]) % q,
      (c1[2] * c2[0]) % q, (c1[2] * c2[1]) % q, (c1[2] * c2[2]) % q]

# phase(곱) = 두 텐서의 내적 (9항 직접)
ph_mul = center(cm[0]*t[0] + cm[1]*t[1] + cm[2]*t[2]
              + cm[3]*t[3] + cm[4]*t[4] + cm[5]*t[5]
              + cm[6]*t[6] + cm[7]*t[7] + cm[8]*t[8])

print("텐서곱 차원:", len(cm), " / phase(곱) =", ph_mul, " ≈ Delta^2*2*3 =", Delta**2 * 6)
print("Delta^2 로 나눠 복호:", round(ph_mul / Delta**2), "(기대 6)")
assert round(ph_mul / Delta**2) == 2 * 3
print("OK: 동형 곱셈 (2차 암호문)")

### 선택 심화 3-1. 왜 하필 '텐서'인가
$\mu_1\mu_2=(\bar{\mathbf c}_1^\top\bar{\mathbf s})(\bar{\mathbf s}^\top\bar{\mathbf c}_2)
=\bar{\mathbf s}^\top(\bar{\mathbf c}_1\bar{\mathbf c}_2^\top)\bar{\mathbf s}
=\langle\bar{\mathbf c}_1\otimes\bar{\mathbf c}_2,\ \bar{\mathbf s}\otimes\bar{\mathbf s}\rangle$.
$\mu_1\mu_2$ 와 텐서 내적이 같음을 직접 확인.


In [ ]:
mu1 = phase(c1)
mu2 = phase(c2)
# (3)번(텐서 내적)은 위 ph_mul 과 동일해야 함
print("(1) mu1 * mu2      =", center(mu1 * mu2))
print("(3) 텐서 내적       =", ph_mul)
assert center(mu1 * mu2) == ph_mul
print("=> 곱한 phase = 텐서 내적. 텐서곱은 '키에 대한 이차형식'을 편 것.")

### ★ 실습 2. 이제 $n=3$ — 손으로 다 쓰긴 벅차니 **for-루프로 일반화**
위 $n=2$ 는 회로를 전부 펼쳐 썼지만, $n=3$(확장키 4, 텐서 16)부터는 항이 많아
루프로 일반화합니다. `None` 슬롯 세 곳을 루프로 채우면 자동 검증됩니다.


In [ ]:
n3 = 3
r3 = random.Random(7)

def keygen3():
    out = []
    for i in range(n3):
        out.append(r3.randint(-10, 10))
    return out

def encrypt3(m, sk):
    a = []
    for i in range(n3):
        a.append(r3.randint(0, q - 1))
    inner = 0
    for i in range(n3):
        inner = inner + a[i] * sk[i]
    b = (inner + r3.randint(-10, 10) + Delta * m) % q
    ct = [b]
    for i in range(n3):
        ct.append((-a[i]) % q)
    return ct

def dot(u, v):
    total = 0
    for i in range(len(u)):
        total = total + u[i] * v[i]
    return total

s3 = keygen3()
c1_3 = encrypt3(2, s3)
c2_3 = encrypt3(3, s3)

# ---- 채워야 할 슬롯 3개 (for-루프로) --------------------------------------
sbar3 = None   # 확장키 [1, s1, s2, s3]  힌트: r=[1]; for i in range(len(s3)): r.append(s3[i])
t3    = None   # 텐서곱 키  힌트: 이중 for 로 sbar3[i]*sbar3[j] 를 append
cmul3 = None   # 암호문 텐서곱  힌트: 이중 for 로 (c1_3[i]*c2_3[j])%q 를 append
# --------------------------------------------------------------------------

if sbar3 is None or t3 is None or cmul3 is None:
    print("슬롯(None)을 for-루프로 채운 뒤 다시 실행하세요.")
    print("기대값 -> 확장키 차원 4, 텐서 차원 16, phase(곱) 복호 6")
else:
    add3 = []
    for i in range(len(c1_3)):
        add3.append((c1_3[i] + c2_3[i]) % q)
    ph_add = center(dot(add3, sbar3))
    ph_m3  = center(dot(cmul3, t3))
    print("확장키 차원:", len(sbar3), "(기대 4) / 텐서 차원:", len(t3), "(기대 16)")
    print("덧셈 복호 :", round(ph_add / Delta), "(기대 5)")
    print("곱셈 복호 :", round(ph_m3 / Delta**2), "(기대 6)")
    assert len(sbar3) == 4 and len(t3) == 16
    assert round(ph_add / Delta) == 5 and round(ph_m3 / Delta**2) == 6
    print("OK: n=3 에서도 동작 🎉  (n 이 커지면 루프가 필수!)")

## 4. 리니어라이제이션 (전부 직접, 단일 비밀 $s$)
곱셈 후 새 비밀 단항식(여기선 $s^2$)이 생김. 이를 없애 다시 $(1,s)$ 로.
모든 암호문·수를 숫자로 직접 씁니다 (가젯 $B=10$, 자리 2개).


In [ ]:
sk = 2                      # 이 예에선 단일 비밀 s = 2
# 곱셈 직후 2차 암호문 (키 1, s, s^2 아래):  d = (d0, d1, d2)
d0, d1, d2 = 3, 18, 24      # phase = 3 + 18*s + 24*s^2 = 3 + 36 + 96 = 135

# KSK: s^2(=4) 를 각 10진 자리로 암호화 (암호문 = (c0, c1), phase = c0 + c1*s)
KSK0 = (2, 1)               # Enc_s(4*1)  : phase 2 + 1*s = 4
KSK1 = (30, 5)              # Enc_s(4*10) : phase 30 + 5*s = 40

# d2 = 24 = 4*1 + 2*10  ->  자릿수 [4, 2]
# 결합: 4*KSK0 + 2*KSK1  (두 성분 직접)
kb = 4 * KSK0[0] + 2 * KSK1[0]      # 4*2 + 2*30 = 68
ka = 4 * KSK0[1] + 2 * KSK1[1]      # 4*1 + 2*5  = 14

# 최종 relin 암호문 (키 1, s):  s^2 항이 사라짐
c_relin = [(d0 + kb) % q, (d1 + ka) % q]     # (3+68, 18+14) = (71, 32)
ph = center(c_relin[0] + c_relin[1] * sk)    # phase = c0 + c1*s
print("relin 후 암호문 =", c_relin, ", phase =", ph, " (원래 135 보존, s^2 제거)")
assert ph == 3 + 18 * sk + 24 * (sk * sk)
print("OK: 리니어라이제이션 (phase 보존)")

## 5. 리스케일 — $\div\Delta$ 로 스케일 되돌리기 (직접)
곱셈 후 스케일이 $\Delta^2$ 이니 $\Delta$ 로 나눠 $\Delta$ 로 되돌립니다.


In [ ]:
ph_before = 60496                 # 곱 직후 phase (스케일 Δ^2 = 10^4)
ph_after  = round(ph_before / Delta)   # ÷Δ  ->  605  (스케일 Δ = 10^2)
print("곱 직후 phase =", ph_before, " (스케일 10^4)")
print("÷Δ 후 phase   =", ph_after,  " (스케일 10^2)  -> 복호:", round(ph_after / Delta), "(기대 6)")
assert round(ph_after / Delta) == 6
print("OK: 리스케일 -> 원래 스케일 (곱셈 결과 6)")

# 2부. NTT — 다항식 곱셈을 빠르게
링 $\mathbb Z_{17}[X]/(X^4-1)$ 에서 순환 합성곱을 **스쿨북 = FFT = NTT** 로.
$N=4$ 라 4점 변환을 전부 펼칩니다.


In [ ]:
P = 17
a = [1, 2, 3, 4]
b = [5, 6, 7, 8]
print("4^1..4^4 mod 17 =", [pow(4, 1, P), pow(4, 2, P), pow(4, 3, P), pow(4, 4, P)], "(원시 4차 근)")

## 6. 스쿨북 순환 합성곱 ($X^4\equiv1$ 이라 인덱스 $\bmod 4$, 네 출력 직접)

In [ ]:
# res[k] = 합 (i+j ≡ k mod 4) a[i]*b[j]   — 네 개를 그대로
s0 = (a[0]*b[0] + a[1]*b[3] + a[2]*b[2] + a[3]*b[1]) % P
s1_ = (a[0]*b[1] + a[1]*b[0] + a[2]*b[3] + a[3]*b[2]) % P
s2_ = (a[0]*b[2] + a[1]*b[1] + a[2]*b[0] + a[3]*b[3]) % P
s3_ = (a[0]*b[3] + a[1]*b[2] + a[2]*b[1] + a[3]*b[0]) % P
c_school = [s0, s1_, s2_, s3_]
print("스쿨북 순환 합성곱 mod 17 =", c_school)

## 7. FFT 로 같은 결과 (4점 DFT, 트위들 $1,-i,-1,i$ 직접)

In [ ]:
def dft4(x):
    # X[k] = 합 x[j]*exp(-2πi jk/4);  트위들: 1, -i, -1, i
    X0 = x[0] + x[1] + x[2] + x[3]
    X1 = x[0] - 1j*x[1] - x[2] + 1j*x[3]
    X2 = x[0] - x[1] + x[2] - x[3]
    X3 = x[0] + 1j*x[1] - x[2] - 1j*x[3]
    return [X0, X1, X2, X3]

def idft4(X):
    # x[j] = (1/4) 합 X[k]*exp(+2πi jk/4)
    x0 = (X[0] + X[1] + X[2] + X[3]) / 4
    x1 = (X[0] + 1j*X[1] - X[2] - 1j*X[3]) / 4
    x2 = (X[0] - X[1] + X[2] - X[3]) / 4
    x3 = (X[0] - 1j*X[1] - X[2] + 1j*X[3]) / 4
    return [x0, x1, x2, x3]

Fa = dft4(a)
Fb = dft4(b)
Fp = [Fa[0]*Fb[0], Fa[1]*Fb[1], Fa[2]*Fb[2], Fa[3]*Fb[3]]   # 포인트와이즈 곱
inv = idft4(Fp)
c_fft = [round(inv[0].real) % P, round(inv[1].real) % P,
         round(inv[2].real) % P, round(inv[3].real) % P]
print("FFT 순환 합성곱 mod 17   =", c_fft)
assert c_fft == c_school
print("OK: 스쿨북 == FFT")

## 8. 원시근 4로 만든 NTT (정수 mod 17, 4점 변환 직접)
$W_{ij}=4^{ij}\bmod17$. 행이 $[1,1,1,1],[1,4,16,13],[1,16,1,16],[1,13,16,4]$.
역변환은 $w^{-1}=13$, $N^{-1}=13$.


In [ ]:
def ntt(x):
    A0 = (x[0] +      x[1] +      x[2] +      x[3]) % P
    A1 = (x[0] +  4 * x[1] + 16 * x[2] + 13 * x[3]) % P
    A2 = (x[0] + 16 * x[1] +      x[2] + 16 * x[3]) % P
    A3 = (x[0] + 13 * x[1] + 16 * x[2] +  4 * x[3]) % P
    return [A0, A1, A2, A3]

def intt(X):
    # 역행렬(w^-1=13) 곱한 뒤 N^-1=13 배
    a0 = (13 * (X[0] +      X[1] +      X[2] +      X[3])) % P
    a1 = (13 * (X[0] + 13 * X[1] + 16 * X[2] +  4 * X[3])) % P
    a2 = (13 * (X[0] + 16 * X[1] +      X[2] + 16 * X[3])) % P
    a3 = (13 * (X[0] +  4 * X[1] + 16 * X[2] + 13 * X[3])) % P
    return [a0, a1, a2, a3]

Na = ntt(a)
Nb = ntt(b)
Np = [(Na[0]*Nb[0]) % P, (Na[1]*Nb[1]) % P, (Na[2]*Nb[2]) % P, (Na[3]*Nb[3]) % P]
c_ntt = intt(Np)
print("NTT 순환 합성곱 mod 17   =", c_ntt)
assert c_ntt == c_school
print("OK: NTT == 스쿨북 == FFT")

## 9. 주파수 영역 mod 17 왕복

In [ ]:
A = ntt(a)
back = intt(A)
a_mod = [a[0] % P, a[1] % P, a[2] % P, a[3] % P]
print("NTT(a)       =", A)
print("INTT(NTT(a)) =", back, " (원래 a =", a_mod, ")")
assert back == a_mod
print("OK: NTT <-> INTT 왕복 항등 (mod 17)")

### ★ 실습 3. 단위근을 바꾸면?
`TRY_W` 를 바꿔 4점 NTT 를 만들고 왕복이 되는지 보세요.
`TRY_W=4` 는 원시 4차 근(차수 4). `TRY_W=2` 면 차수가 8 이라 왕복이 깨집니다.


In [ ]:
TRY_W = 4

# TRY_W 의 거듭제곱 4개를 직접: 원시 4차 근이면 ^4 에서 처음 1
p1 = pow(TRY_W, 1, P)
p2 = pow(TRY_W, 2, P)
p3 = pow(TRY_W, 3, P)
p4 = pow(TRY_W, 4, P)
is_primitive4 = (p4 == 1 and p2 != 1 and p1 != 1)

winv = pow(TRY_W, -1, P)
Ninv = pow(4, -1, P)

# 정변환 (행마다 4^(ij) — 4점 직접)
def W(i, j):
    return pow(TRY_W, i * j, P)
A0 = (W(0,0)*a[0] + W(0,1)*a[1] + W(0,2)*a[2] + W(0,3)*a[3]) % P
A1 = (W(1,0)*a[0] + W(1,1)*a[1] + W(1,2)*a[2] + W(1,3)*a[3]) % P
A2 = (W(2,0)*a[0] + W(2,1)*a[1] + W(2,2)*a[2] + W(2,3)*a[3]) % P
A3 = (W(3,0)*a[0] + W(3,1)*a[1] + W(3,2)*a[2] + W(3,3)*a[3]) % P
Ax = [A0, A1, A2, A3]

def Wi(i, j):
    return pow(winv, i * j, P)
b0 = (Ninv*(Wi(0,0)*A0 + Wi(0,1)*A1 + Wi(0,2)*A2 + Wi(0,3)*A3)) % P
b1 = (Ninv*(Wi(1,0)*A0 + Wi(1,1)*A1 + Wi(1,2)*A2 + Wi(1,3)*A3)) % P
b2 = (Ninv*(Wi(2,0)*A0 + Wi(2,1)*A1 + Wi(2,2)*A2 + Wi(2,3)*A3)) % P
b3 = (Ninv*(Wi(3,0)*A0 + Wi(3,1)*A1 + Wi(3,2)*A2 + Wi(3,3)*A3)) % P
back = [b0, b1, b2, b3]
a_mod = [a[0] % P, a[1] % P, a[2] % P, a[3] % P]

print("TRY_W 거듭제곱 [^1,^2,^3,^4]:", [p1, p2, p3, p4])
print("원시 4차 근인가? (^4 에서 처음 1)", is_primitive4)
print("왕복 결과:", back, " / 원래:", a_mod)
print("왕복 성공?", back == a_mod)

### 정리
- 순환 합성곱은 **스쿨북 = FFT = NTT** 로 모두 같은 결과.
- 이 노트북은 토이 크기라 **회로를 전부 펼쳐** 어떤 곱·합이 일어나는지 직접 보였습니다.
- $n,N$ 이 커지면(실전) 이 펼친 식이 **for-루프**가 됩니다 (★실습 2 참고).

**수고하셨습니다! 🎉**
